# TrabajO Semestral GDO\n
## Digital Twin FJSSP con Rush Order\n
\n
**Objetivo:** resolver un Flexible Job Shop Scheduling Problem, simular una perturbacion tipo rush order en `t = Cmax/2` y evaluar una estrategia inteligente de recuperacion.\n
\n
**Paper base:** Liu et al. (2022), *Digital Twin-Driven Adaptive Scheduling for Flexible Job Shops*.

## 0. Control del equipo\n
\n
| Integrante | Rol | Pendiente actual | Estado |\n
|---|---|---|---|\n
| Nombre 1 | Scheduling / optimizacion | | |\n
| Nombre 2 | Perturbacion / Digital Twin | | |\n
| Nombre 3 | XGBoost / metricas | | |\n
| Nombre 4 | Informe / presentacion | | |\n
\n
### Decisiones fijas\n
\n
| Decision | Valor | Justificacion breve |\n
|---|---|---|\n
| Perturbacion | Rush order | Caso dinamico pedido/elegido |\n
| Instante de perturbacion | `Cmax/2` | Requisito del profesor |\n
| Objetivo principal | Minimizar `Cmax` | Metrica principal del paper |\n
| Scheduler base | Pendiente | Debe generar schedule factible |\n
| Modelo inteligente | XGBoost selector de estrategia | Decide rescheduling, no construye el Gantt |

## 1. Setup\n
\n
Cargar librerias, fijar semillas y definir rutas. Mantener esta celda liviana.

In [ ]:
from pathlib import Path\n
import random\n
\n
import numpy as np\n
import pandas as pd\n
import matplotlib.pyplot as plt\n
\n
SEED = 42\n
random.seed(SEED)\n
np.random.seed(SEED)\n
\n
PROJECT_ROOT = Path.cwd()\n
DATA_DIR = PROJECT_ROOT / "data"\n
RESULTS_DIR = PROJECT_ROOT / "results"\n
\n
PROJECT_ROOT

## 2. Datos del paper\n
\n
Tabla base para comparar nuestros resultados. Si usamos instancias generadas por nosotros, aclarar que son *paper-like*, no identicas al paper.

In [ ]:
paper_results = pd.DataFrame([\n
    ["S1_20_10_75_20", 1411.50, 1437.32, 1464.63],\n
    ["S2_20_10_75_50", 1249.41, 1261.00, 1280.08],\n
    ["S3_20_10_90_20", 1213.56, 1220.30, 1245.30],\n
    ["S4_20_10_90_50",  993.15, 1075.19, 1178.12],\n
    ["S5_100_20_75_20", 4036.58, 4103.53, 4142.77],\n
    ["S6_100_20_75_50", 3688.92, 3829.52, 3839.14],\n
    ["S7_100_20_90_20", 3391.53, 3538.60, 3565.44],\n
    ["S8_100_20_90_50", 3154.40, 3247.31, 3266.01],\n
    ["S9_100_20_90_100", 3423.31, 3415.46, 3502.81],\n
], columns=["instance", "RLEGA_avg", "GA_avg", "TS_avg"])\n
\n
paper_results

## 3. Instancias FJSSP\n
\n
Definir o cargar instancias con esta estructura minima:\n
\n
| job_id | operation_id | candidate_machines | processing_times |\n
|---|---:|---|---|\n
| J1 | 1 | `[3, 8]` | `[38, 49]` |\n
\n
Reglas del paper para instancias generadas:\n
- operaciones por job: `U(1,10)`\n
- tiempos de proceso: `U(1,100)`\n
- flexibilidad: `20%`, `50%`, `100%`\n
- utilizacion: `75%`, `90%`

In [ ]:
# TODO: cargar o generar instancia FJSSP.\n
# instance = ...\n
# display(instance.head())

## 4. Scheduling inicial\n
\n
Salida esperada del scheduler:\n
\n
| job_id | operation_id | machine | start | end |\n
|---|---:|---:|---:|---:|\n
\n
Criterio minimo antes de avanzar: el schedule debe pasar el validador de factibilidad.

In [ ]:
# TODO: generar schedule inicial.\n
# schedule_initial = solve_initial_schedule(instance, seed=SEED)\n
# cmax_initial = schedule_initial["end"].max()\n
# cmax_initial

## 5. Validacion de factibilidad\n
\n
Checks obligatorios:\n
- una operacion no empieza antes de terminar la anterior del mismo job;\n
- una maquina no procesa dos operaciones al mismo tiempo;\n
- cada operacion aparece una sola vez;\n
- `end > start`.

In [ ]:
# TODO: validar factibilidad.\n
# assert validate_schedule(schedule_initial, instance)

## 6. Gantt inicial\n
\n
Grafico para informe y presentacion.

In [ ]:
# TODO: graficar Gantt inicial.\n
# plot_gantt(schedule_initial, title=f"Schedule inicial - Cmax={cmax_initial}")

## 7. Perturbacion: Rush Order en `Cmax/2`\n
\n
Congelar estado en `t*`:\n
- operaciones terminadas: no se cambian;\n
- operaciones en curso: no se interrumpen;\n
- operaciones no iniciadas: pueden reprogramarse;\n
- nueva orden: se incorpora como urgente.

In [ ]:
# TODO: inyectar rush order.\n
# t_star = cmax_initial / 2\n
# rush_order = generate_rush_order(...)\n
# schedule_perturbed = apply_right_shift(schedule_initial, rush_order, t_star)\n
# cmax_perturbed = schedule_perturbed["end"].max()\n
# delta_cmax = cmax_perturbed - cmax_initial\n
# t_star, cmax_perturbed, delta_cmax

## 8. Metricas de dano\n
\n
Metricas minimas:\n
- `Delta Cmax = Cmax_perturbado - Cmax_inicial`\n
- nervousness por numero de operaciones modificadas\n
- nervousness por suma de desplazamientos de inicio

In [ ]:
# TODO: calcular metricas de dano.\n
# damage_metrics = compute_damage(schedule_initial, schedule_perturbed)\n
# damage_metrics

## 9. Estrategias de recuperacion\n
\n
Comparar tres estrategias candidatas:\n
\n
| Estrategia | Descripcion | Costo esperado |\n
|---|---|---|\n
| `right_shift` | desplazar sin optimizar | bajo |\n
| `partial_reschedule` | reoptimizar operaciones no iniciadas afectadas | medio |\n
| `full_reschedule` | reoptimizar todas las operaciones no iniciadas | alto |

In [ ]:
# TODO: aplicar estrategias candidatas.\n
# recovery_results = evaluate_recovery_strategies(instance, schedule_initial, rush_order, t_star)\n
# recovery_results

## 10. Dataset para XGBoost\n
\n
Una fila por escenario de perturbacion. La etiqueta es la mejor estrategia segun:\n
1. menor `Cmax`;\n
2. menor nervousness si hay empate;\n
3. menor tiempo computacional si sigue el empate.

In [ ]:
# TODO: construir dataset supervisado.\n
# dataset = build_training_dataset(scenarios)\n
# dataset.head()

## 11. Modelo inteligente: XGBoost\n
\n
El modelo detecta el evento porque recibe el estado post-perturbacion y selecciona estrategia. No genera el schedule.

In [ ]:
# TODO: entrenar clasificador XGBoost.\n
# from xgboost import XGBClassifier\n
# model = XGBClassifier(...)\n
# model.fit(X_train, y_train)

## 12. Evaluacion final\n
\n
Comparar en escenarios no vistos:\n
\n
| Escenario | Cmax inicial | Cmax perturbado | Cmax recuperado | % recuperacion | nervousness | tiempo recovery |\n
|---|---:|---:|---:|---:|---:|---:|

In [ ]:
# TODO: correr evaluacion end-to-end.\n
# final_results = evaluate_end_to_end(test_scenarios, model)\n
# final_results

## 13. Graficos para informe\n
\n
Minimo recomendado:\n
- Gantt inicial;\n
- Gantt perturbado;\n
- Gantt recuperado;\n
- comparacion de `Cmax`;\n
- matriz de confusion XGBoost;\n
- importancia de variables.

In [ ]:
# TODO: generar graficos finales.

## 14. Conclusiones para redactar\n
\n
- El rush order aumento `Cmax` en: pendiente.\n
- La recuperacion redujo el dano en: pendiente.\n
- XGBoost eligio estrategias con menor costo computacional que reoptimizar todo siempre: pendiente.\n
- Limitaciones: instancias no identicas al paper, tamano del dataset, solo se prueba rush order.